# Autoencoder reconstruction demo

The model, training loop, preprocessing, and evaluation live in `src/`. Run them from the repository root:

```powershell
python -m src.train
python -m src.evaluate
```

This notebook only loads the selected checkpoint and visualizes a held-out test example with the recorded evaluation threshold. The anomaly map uses the same channel-mean absolute reconstruction error as evaluation.


In [ ]:
import json
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from torchvision import transforms

from src.config import CHECKPOINT_DIR, DATA_ROOT, RESULTS_DIR, IMAGE_SIZE
from src.data.dataset import MVTecDataset
from src.evaluate import BOTTLE_MEAN, BOTTLE_STD, build_transform
from src.models.bottle_autoencoder import BottleAutoencoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = CHECKPOINT_DIR / "bottle_autoencoder.pth"
RESULTS_PATH = RESULTS_DIR / "bottle.json"


In [ ]:
metadata = json.loads(CHECKPOINT_PATH.with_suffix(".meta.json").read_text(encoding="utf-8"))
evaluation = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
model = BottleAutoencoder(latent_dim=metadata["latent_dim"]).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
model.eval()

mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])
test_data = MVTecDataset(DATA_ROOT, "bottle", "test", transform=build_transform(), mask_transform=mask_transform)
example_index = next(i for i, (_, _, label, _) in enumerate(test_data) if int(label.item()) == 1)
image, mask, _, image_path = test_data[example_index]
with torch.inference_mode():
    reconstruction = model(image.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
anomaly_map = gaussian_filter(torch.abs(image - reconstruction).mean(dim=0).numpy(), sigma=evaluation["sigma"])
binary_map = anomaly_map >= evaluation["threshold"]
mean = torch.tensor(BOTTLE_MEAN).view(3, 1, 1)
std = torch.tensor(BOTTLE_STD).view(3, 1, 1)
original_rgb = ((image * std + mean).clamp(0, 1)).permute(1, 2, 0).numpy()
reconstruction_rgb = ((reconstruction * std + mean).clamp(0, 1)).permute(1, 2, 0).numpy()
print(f"Example: {image_path}")
print(f"Validation-derived threshold: {evaluation['threshold']:.5f}; sigma: {evaluation['sigma']}")


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(original_rgb); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(reconstruction_rgb); axes[1].set_title("Reconstruction"); axes[1].axis("off")
im = axes[2].imshow(anomaly_map, cmap="inferno"); axes[2].set_title("Mean absolute error"); axes[2].axis("off")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[3].imshow(binary_map, cmap="gray"); axes[3].set_title("Validation threshold"); axes[3].axis("off")
axes[4].imshow(mask.squeeze().numpy(), cmap="gray"); axes[4].set_title("Ground truth"); axes[4].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
{key: evaluation[key] for key in ("pixel_auroc", "image_auroc", "pro_auc_max_fpr_0_30", "threshold", "sigma", "loss")}
